# 1. Setup
**Approach 2: Feature disentanglement.** The backbone feature is split into a crop part and a disease part.
Adversarial heads (with gradient reversal) try to read the disease from the crop part, and the crop
from the disease part. The encoders learn to defeat them, so each part keeps only its own concept.
An orthogonality loss also stops the two parts from carrying the same information.
Idea: if disease features hold no crop information, a seen disease is easier to spot on a new crop.

In [ ]:
import copy
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # lets the notebook import common/

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

from common.config import CHECKPOINT_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, set_seed
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

MODEL_NAME = "approach2_disentangle"
DEBUG = True  # True: small data, 1 epoch. Set False for the full run.
HP = {"backbone": "resnet50", "lr": 1e-4, "weight_decay": 1e-4, "epochs": 20,
      "batch_size": 64, "optimizer": "AdamW", "label_smoothing": 0.1, "concept_dim": 512,
      "adversarial_weight": 0.3, "orthogonal_weight": 0.1}
EPOCHS = 1 if DEBUG else HP["epochs"]
set_seed(42)

# 2. Data

In [ ]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="imagenet", debug=DEBUG)
pairs = get_pairs().to(DEVICE)

# 3. Model

In [ ]:
class GradientReversal(torch.autograd.Function):
    """Identity going forward, flips the gradient going back (Ganin and Lempitsky, 2015)."""

    @staticmethod
    def forward(ctx, features):
        return features.view_as(features)

    @staticmethod
    def backward(ctx, grad):
        return -grad


class DisentangleNet(nn.Module):
    def __init__(self, backbone_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        width, dim = self.backbone.num_features, HP["concept_dim"]
        self.crop_encoder = nn.Sequential(nn.Linear(width, dim), nn.ReLU())
        self.disease_encoder = nn.Sequential(nn.Linear(width, dim), nn.ReLU())
        self.crop_head, self.disease_head = nn.Linear(dim, NUM_CROPS), nn.Linear(dim, NUM_DISEASES)
        self.disease_from_crop = nn.Linear(dim, NUM_DISEASES)  # adversary
        self.crop_from_disease = nn.Linear(dim, NUM_CROPS)  # adversary

    def forward(self, images):
        features = self.backbone(images)
        crop_features, disease_features = self.crop_encoder(features), self.disease_encoder(features)
        logits = (self.crop_head(crop_features), self.disease_head(disease_features))
        adversary_logits = (self.disease_from_crop(GradientReversal.apply(crop_features)),
                            self.crop_from_disease(GradientReversal.apply(disease_features)))
        return logits, adversary_logits, (crop_features, disease_features)


def orthogonal_loss(crop_features, disease_features):
    """Mean squared cross-correlation between the two parts. Zero when they are unrelated."""
    crop_norm = F.normalize(crop_features - crop_features.mean(0), dim=0)
    disease_norm = F.normalize(disease_features - disease_features.mean(0), dim=0)
    return (crop_norm.T @ disease_norm).pow(2).mean()


model = DisentangleNet(HP["backbone"]).to(DEVICE)


@torch.no_grad()
def predict(images):
    model.eval()
    (crop_logits, disease_logits), _, _ = model(images.to(DEVICE))
    crop_log_prob = F.log_softmax(crop_logits.float(), dim=1)
    disease_log_prob = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log_prob[:, pairs[:, 0]] + disease_log_prob[:, pairs[:, 1]]

# 4. Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=HP["label_smoothing"])
best_val, best_state = -1.0, None

for epoch in range(EPOCHS):
    model.train()
    for images, crop_labels, disease_labels, _ in train_loader:
        images = images.to(DEVICE)
        crop_labels, disease_labels = crop_labels.to(DEVICE), disease_labels.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            (crop_logits, disease_logits), (adv_disease, adv_crop), parts = model(images)
            ce_loss = criterion(crop_logits, crop_labels) + criterion(disease_logits, disease_labels)
            adversary_loss = criterion(adv_disease, disease_labels) + criterion(adv_crop, crop_labels)
        ortho_loss = orthogonal_loss(*[part.float() for part in parts])
        loss = ce_loss + HP["adversarial_weight"] * adversary_loss + HP["orthogonal_weight"] * ortho_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()
    val_top1 = val_accuracy(predict, val_loader)
    print(f"epoch {epoch + 1}: CE {ce_loss.item():.3f}, adversary {adversary_loss.item():.3f}, "
          f"orthogonal {ortho_loss.item():.4f}, val top-1 {val_top1:.2f}%")
    if val_top1 > best_val:  # keep the best epoch by val, never by test
        best_val, best_state = val_top1, copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)

# 5. Evaluation

In [ ]:
results = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, HP, robust=True)

# 6. Save

In [ ]:
CHECKPOINT_DIR.mkdir(exist_ok=True)
torch.save({"model_name": MODEL_NAME, "state_dict": model.state_dict(), "hp": HP},
           CHECKPOINT_DIR / f"{MODEL_NAME}.pt")